Mascara comparison: from single black shade to three colored options.
*Co-authored with CoCo*

In [ ]:
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt

for weight_file in ['ClashGrotesk-Extralight.ttf', 'ClashGrotesk-Light.ttf',
                    'ClashGrotesk-Regular.ttf', 'ClashGrotesk-Medium.ttf',
                    'ClashGrotesk-Semibold.ttf', 'ClashGrotesk-Bold.ttf']:
    fm.fontManager.addfont(f'fonts/Clash_Grotesk/{weight_file}')

plt.rcParams['font.family'] = 'Clash Grotesk'
plt.rcParams['font.weight'] = 'light'

In [ ]:
%%sql -r db_context
USE DATABASE MARCJACOBS_BEAUTY_REVIVAL;

In [ ]:
%%sql -r mascara_data
SELECT
  s.era,
  s.product_name,
  s.shade_name,
  s.shade_description,
  s.finish,
  s.c_pct,
  s.m_pct,
  s.y_pct,
  s.k_pct,
  p.price_usd,
  p.size_oz,
  p.shade_count,
  p.hero_claims
FROM MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.SHADES s
JOIN MARCJACOBS_BEAUTY_REVIVAL.ANALYTICS.PRODUCTS p
  ON s.era = p.era AND s.product_name = p.product_name
WHERE s.product_type = 'Mascara'
  AND s.shade_name IS NOT NULL
ORDER BY s.era, s.shade_name;

In [ ]:
import pandas as pd
import colorsys
from matplotlib.lines import Line2D
import numpy as np

def cmyk_to_rgb(c, m, y, k):
    c, m, y, k = c / 100, m / 100, y / 100, k / 100
    r = (1 - c) * (1 - k)
    g = (1 - m) * (1 - k)
    b = (1 - y) * (1 - k)
    return (r, g, b)

def rgb_to_hsl(r, g, b):
    h, l, s = colorsys.rgb_to_hls(r, g, b)
    return h * 360, l * 100, s * 100

df = mascara_data.to_pandas() if hasattr(mascara_data, 'to_pandas') else mascara_data
df = df.copy()
df.columns = [col.lower() for col in df.columns]

df['rgb'] = df.apply(lambda row: cmyk_to_rgb(row['c_pct'], row['m_pct'], row['y_pct'], row['k_pct']), axis=1)
df['hue'] = df['rgb'].apply(lambda rgb: rgb_to_hsl(*rgb)[0])
df['lightness'] = df['rgb'].apply(lambda rgb: rgb_to_hsl(*rgb)[1])

# Original mascara was just black (no shade data in table)
original_black = {'shade_name': 'Blacquer (single shade)', 'hue': 0, 'lightness': 4,
                  'rgb': cmyk_to_rgb(0, 0, 0, 96)}

era_labels = {'original_launch': '2013 Collection', 'revival': '2026 Revival'}
era_annotations = {
    'original_launch': 'Single volumizing black mascara',
    'revival': '3 colored shades: Onyx, Cocoa, and Indigo',
}

def plot_mascara(layout='horizontal'):
    era_order = ['original_launch', 'revival']
    if layout == 'horizontal':
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)
    else:
        fig, axes = plt.subplots(2, 1, figsize=(9, 12), sharex=True)

    for i, (axis, era) in enumerate(zip(axes.flat, era_order)):
        if era == 'original_launch':
            axis.scatter(
                [original_black['hue']], [original_black['lightness']],
                s=200, alpha=0.95, c=[original_black['rgb']],
                edgecolors='black', linewidths=0.6,
            )
            axis.annotate(
                original_black['shade_name'],
                (original_black['hue'], original_black['lightness']),
                textcoords='offset points', xytext=(8, 0),
                fontsize=9, fontweight='normal', alpha=0.85, va='center',
            )
        else:
            era_df = df[df['era'] == era].copy()
            colors = list(era_df['rgb'])
            axis.scatter(
                era_df['hue'], era_df['lightness'],
                s=200, alpha=0.95, c=colors,
                edgecolors='black', linewidths=0.6,
            )
            for _, row in era_df.iterrows():
                axis.annotate(
                    row['shade_name'],
                    (row['hue'], row['lightness']),
                    textcoords='offset points', xytext=(8, 0),
                    fontsize=9, fontweight='normal', alpha=0.85, va='center',
                )

        axis.set_title(era_labels[era], fontsize=13, fontweight='medium', pad=20)
        axis.text(0.5, 1.01, era_annotations[era],
                  transform=axis.transAxes, ha='center', fontsize=11,
                  fontweight='normal', color='#444')
        axis.set_xlabel('Hue (color family)', fontsize=11, fontweight='normal')
        axis.set_xlim(-20, 380)
        axis.set_ylim(-5, 55)
        axis.grid(alpha=0.2)
        axis.tick_params(labelsize=9)

        if layout == 'vertical' or i == 0:
            axis.set_ylabel('Lightness (%)', fontsize=11, fontweight='normal')

    if layout == 'horizontal':
        fig.suptitle('MASCARA: FROM ONE BLACK TO THREE COLORED SHADES',
                     fontsize=18, fontweight='medium', y=0.95)
        plt.tight_layout()
    else:
        plt.tight_layout()
        plt.subplots_adjust(hspace=0.22, top=0.94)
        fig.suptitle('MASCARA: FROM ONE BLACK TO THREE COLORED SHADES',
                     fontsize=18, fontweight='medium', y=1.05)
    plt.show()

plot_mascara('horizontal')
plot_mascara('vertical')